# Run matrix — cola de experimentos

Notebook para Colab que define listas de `ExperimentConfig` y las corre secuencialmente con `run_matrix()`: salta las que ya estan completas, reanuda (sin borrar nada) las que quedaron incompletas de una corrida anterior -- solo recalculando, region por region, lo que falte o este invalido (ver `docs/CHECKPOINT_RESUME.md`) --, continua si una falla, y al final muestra un resumen de completadas/fallidas y MAPE promedio.

## Tres fases, independientes entre si

Este notebook queda organizado en **3 fases**, cada una en su propio bloque de celdas (config + lanzamiento). **Correr una fase NO dispara las otras** -- son bloques de celdas separados, cada uno arma su propia lista de `ExperimentConfig` y llama a `run_matrix()` por su cuenta. Podes correr solo la celda de Setup + solo la fase que te interese.

- **FASE 1A -- Modelos principales**: `xgboost`, `lightgbm`, `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`. Es la matriz baseline ya confirmada como 6/6 completa en Drive -- las configs de esta fase son **exactamente** las que ya se corrieron (mismas exogenas, mismo `train_hours`), sin ningun cambio. Volver a correr esta fase debe detectar todo como completo y saltarlo entero (ver la nota en su seccion).
- **FASE 1B -- Modelos adicionales / baselines**: `naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Los 6 modelos migrados en la tarea de checkpoint/resume, cada uno con su config vigente en `runner.MODEL_DEFAULTS` (sin inventar ningun parametro nuevo).
- **FASE 2 -- Exogenas individuales**: para cada modelo **multivariado** (el que tenga al menos una exogena en su catalogo), una corrida por cada una de `Temperatura, IGAE, Generacion, Importacion, Exportacion`, **sin acumular** (nunca dos exogenas juntas). Generada automaticamente con `build_individual_exog_matrix()` -- no hay que escribir cada `ExperimentConfig` a mano. Los modelos univariados se excluyen solos (no generan corridas redundantes).

Ninguna fase modifica ni borra los resultados de las otras: cada `ExperimentConfig` distinto (modelo + exogenas + train_hours + forecast_horizon) cae en su propio `RUN_NAME`/carpeta determinista (`build_run_name()`, ver `config.py`), asi que 1A/1B/2 nunca se pisan entre si ni pisan lo que ya hay en Drive.

## Datos de entrada

Los 34 archivos (`IGAE_2.xlsx`, `Temperaturas promedio.csv`, y por cada una de las 8 regiones -- BCA, CEN, NES, NOR, NTE, OCC, ORI, PEN -- `{REGION}_long.csv`, `{REGION}_GEN.csv`, `{REGION}_IMP.csv`, `{REGION}_EXP.csv`) viven permanentemente en Google Drive, en `MyDrive/Bases de datos Tesis` (constante `DATA_DIR` mas abajo). Detalle exacto de columnas/formato: [`docs/DATOS_REQUERIDOS.md`](../docs/DATOS_REQUERIDOS.md).

## Modelos disponibles (los 12 registrados hoy en `runner.MODEL_DEFAULTS`/`MODEL_RUNNERS`)

`xgboost`, `lightgbm` (adaptado, ver [`docs/MODELOS_MIGRADOS.md`](../docs/MODELOS_MIGRADOS.md)), `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`, `naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Todos procesan las 8 regiones en una sola corrida.

## Setup (una sola celda): montar Drive, instalar dependencias, cargar el proyecto

Cubre las dependencias de las 3 fases (`optuna`/`lightgbm`, que no vienen preinstaladas en el runtime estandar de Colab; `tensorflow`/`statsmodels`/`xgboost`/`pandas`/`numpy`/`scikit-learn` si vienen). Corre esta celda una sola vez por sesion, sin importar que fase(s) vayas a lanzar despues.

In [ ]:
import os
import sys

from google.colab import drive
drive.mount("/content/drive")

# xgboost, tensorflow, statsmodels, pandas, numpy, scikit-learn ya vienen
# preinstalados en el runtime estandar de Colab; optuna y lightgbm no.
!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("\nSetup completo.")
print("SRC_DIR en sys.path:", SRC_DIR in sys.path)

In [ ]:
# Ruta donde viven permanentemente los datos de entrada en Google Drive.
# Compartida por las 3 fases -- ninguna la sobreescribe ni la necesita
# distinta.
DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"

from tesis_forecast.config import ExperimentConfig
from tesis_forecast.matrix import build_individual_exog_matrix, run_matrix, resumen_dataframe

---
## FASE 1A -- Modelos principales

`xgboost`, `lightgbm`, `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`. **Estas son exactamente las configs que ya se corrieron** para esta matriz (mismas exogenas, mismo `train_hours` que la corrida ya confirmada como 6/6 completa en Drive) -- no se cambio nada aqui, solo se le puso nombre de fase.

Deja `exogenas`/`train_hours` explicitos (en vez de `None`) a proposito, para que quede documentado en el propio notebook cual es la config real de cada corrida de esta fase, no solo "el default de turno" del modulo.

**Reanudar/re-lanzar esta fase**: si volves a correr la celda de "Lanzar FASE 1A" sobre una matriz ya completa, `run_matrix()` llama `validar_resultado(run_dir)` por cada config *antes* de tocar nada -- si encuentra las 8 regiones con metricas validas, imprime `YA COMPLETO, se salta: <RUN_NAME>` y pasa a la siguiente config sin ejecutar ningun modelo. Si alguna quedara incompleta (por ejemplo la sesion de Colab se desconecto a mitad de una region), la reanuda en la misma carpeta -- el checkpoint por region (`checkpoint.cargar_checkpoint_regiones`, invocado dentro de cada `run()`) salta las regiones que ya tengan resultado valido y solo recalcula las que falten. Ningun resultado existente se borra ni se sobreescribe por volver a correr esta celda.

In [ ]:
configs_1a = [
    ExperimentConfig(
        modelo="xgboost",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Fase 1A -- config vigente.",
    ),
    ExperimentConfig(
        modelo="lightgbm",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Fase 1A -- adaptado desde celda 46, no extraccion exacta.",
    ),
    ExperimentConfig(
        modelo="lstm_direct",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=2160,
        notas="Fase 1A -- config vigente.",
    ),
    ExperimentConfig(
        modelo="sarimax",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=1440,
        notas="Fase 1A -- config vigente, orden SARIMAX fijo (sin tuning).",
    ),
    ExperimentConfig(
        modelo="fcnn",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Fase 1A -- config vigente, produce 2 modelos por region (directa + STL-residuos).",
    ),
    ExperimentConfig(
        modelo="ensemble_stl",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Fase 1A -- config vigente, es el pipeline mas pesado (2 redes + barrido AR por region).",
    ),
]

print(f"FASE 1A: {len(configs_1a)} configs")
configs_1a

### Lanzar FASE 1A

Corre secuencialmente. Puede tardar horas (Ensemble y FCNN entrenan redes por cada una de las 8 regiones). Si ya esta todo completo en Drive, esta celda debe correr rapido e imprimir `Saltadas: 6 (ya estaban completas)` en el resumen final, sin re-entrenar nada.

In [ ]:
resultados_1a = run_matrix(configs_1a, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_1a)

---
## FASE 1B -- Modelos adicionales / baselines

`naive`, `naive_trend`, `naive_trend_seasonal`, `ar`, `ar_resid_trend_seasonal`, `lstm_resid`. Los 6 modelos migrados en la tarea de checkpoint/resume (ver `docs/CHECKPOINT_RESUME.md` y `docs/MODELOS_MIGRADOS.md`).

`exogenas`/`train_hours`/`forecast_horizon`/`optuna_n_trials` se dejan en `None` a proposito: esa es la señal para que `resolve_run()`/`run_experiment()` usen exactamente lo que ya esta registrado en `runner.MODEL_DEFAULTS` para cada modelo -- nada se inventa ni se fuerza aqui. Los primeros 5 son univariados (`catalogo == []` en su modulo, `resolve_run()` rechaza cualquier exogena que se les pase); `lstm_resid` es multivariado y usa su catalogo de 5 exogenas por defecto (`Temperatura, IGAE, Generacion, Importacion, Exportacion`).

Misma logica de reanudacion/checkpoint que la Fase 1A: una config ya completa se salta, una incompleta se reanuda solo desde las regiones que falten.

In [ ]:
configs_1b = [
    ExperimentConfig(modelo="naive",
                      notas="Fase 1B -- Naive."),
    ExperimentConfig(modelo="naive_trend",
                      notas="Fase 1B -- Naive + Tendencia."),
    ExperimentConfig(modelo="naive_trend_seasonal",
                      notas="Fase 1B -- Naive + Tendencia + Estacionalidad."),
    ExperimentConfig(modelo="ar",
                      notas="Fase 1B -- AR standalone."),
    ExperimentConfig(modelo="ar_resid_trend_seasonal",
                      notas="Fase 1B -- AR sobre residuos + Tendencia + Estacionalidad."),
    ExperimentConfig(modelo="lstm_resid",
                      notas="Fase 1B -- LSTM multivariada sobre residuos + Tendencia + Estacionalidad."),
]

print(f"FASE 1B: {len(configs_1b)} configs")
configs_1b

### Lanzar FASE 1B

In [ ]:
resultados_1b = run_matrix(configs_1b, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_1b)

---
## FASE 2 -- Exogenas individuales

Para cada modelo **multivariado** (el que tenga al menos una exogena en su catalogo -- ver `runner.MODEL_DEFAULTS`), una corrida por cada exogena de `EXOGENAS_INDIVIDUALES`, **sin acumular** (nunca dos exogenas juntas en la misma corrida). Los modelos univariados (hoy: `naive`, `naive_trend`, `ar`, `naive_trend_seasonal`, `ar_resid_trend_seasonal`) se excluyen automaticamente -- no generan corridas redundantes con una sola exogena c/u.

Cada corrida usa exactamente los defaults cientificos vigentes del modelo (`train_hours`/`forecast_horizon`/`optuna_n_trials`, ver `runner.MODEL_DEFAULTS`); lo UNICO que cambia respecto a las Fases 1A/1B es `exogenas`. El `RUN_NAME` de cada corrida queda en una carpeta propia (ej. `XGBoost_train336h_fh168h_Temp` vs. `XGBoost_train336h_fh168h_IGAE` vs. el baseline de la Fase 1A `XGBoost_train336h_fh168h_Temp-Prim-Sec-Terc-IGAE-Gen-Imp-Exp`), asi que esta fase **no toca ni sobreescribe** los resultados de 1A ni 1B. El checkpoint por region aplica exactamente igual: una corrida individual completa se salta, una incompleta se reanuda solo desde las regiones faltantes.

Ver `tests/test_individual_exog_matrix.py` para la verificacion (numero de configs, sin colisiones de RUN_NAME, defaults preservados, exclusiones documentadas, etc.).

In [ ]:
EXOGENAS_INDIVIDUALES = [
    "Temperatura",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]

configs_individuales = build_individual_exog_matrix(exogenas=EXOGENAS_INDIVIDUALES)

print(f"FASE 2: {len(configs_individuales)} configs")
print("\nPor modelo:")
for modelo in sorted({c.modelo for c in configs_individuales}):
    exogenas_de_modelo = sorted(c.exogenas[0] for c in configs_individuales if c.modelo == modelo)
    print(f"  {modelo}: {exogenas_de_modelo}")

configs_individuales

### Lanzar FASE 2

In [ ]:
resultados_individuales = run_matrix(configs_individuales, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados_individuales)